In [1]:
import os, shutil, rasterio, sys
sys.path.append('backend/app/')
from rasterio.features import shapes, rasterize
from rasterio.mask import mask
from shapely.geometry import shape
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions
from shapely.geometry import Polygon, MultiPolygon
import rioxarray as rioxr
from hydromt_wflow import WflowSbmModel
# from hydromt import initialize_logging
np.random.seed(42)
# initialize_logging()

c:\Envs\hyd_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Functions
def keep_polygon(geom):
    if geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if len(polys) == 0: return None
        return polys[0]
    return geom

def fix_invalid_polygon(gdf, cols):
    gdf_new, name = gdf.copy(), cols[0]
    gdf_valid, gdf_nan = gdf_new[gdf_new[name] != ''], gdf_new[gdf_new[name] == '']
    if gdf_nan.shape[0] > 0:
        gdf_valid['geometry'] = gdf_valid['geometry'].apply(keep_polygon)
        gdf_nan['geometry'] = gdf_nan['geometry'].apply(keep_polygon)
        # Spatial join nearest
        gdf_filled = gpd.sjoin_nearest(
            gdf_nan, gdf_valid[['geometry', name]], how='left', distance_col='dist'
        )
        gdf_filled = gdf_filled.drop_duplicates(subset='_id')
        gdf_new.loc[gdf_filled.index, cols] = gdf_valid.loc[gdf_filled['index_right'], cols].values
    gdf_new['geometry'] = gdf_new['geometry'].apply(keep_polygon)
    return gdf_new

def clip_catchment(catchment, raster_path, out_path, inside=False):
    with rasterio.open(raster_path) as src:
        geoms = catchment.geometry.values
        # Clip raster
        out_image, out_transform = mask(
            src, geoms, crop=False, nodata=-9999, invert=inside
        )
        out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform, "nodata": -9999
    })
    with rasterio.open(out_path, "w", **out_meta) as dest:
        dest.write(out_image)
    del dest

def write_tif(path, terrain, geo, col=''):
    transform = terrain.transform
    if col == '': shapes = ((geom, 1) for geom in geo.geometry)
    else: shapes = ((geom, value) for geom, value in zip(geo.geometry, geo[col]))
    raster = rasterize(
        shapes=shapes, out_shape=(terrain.height, terrain.width),
        transform=transform, fill=-9999, dtype="float32", all_touched=True
    )
    meta = {
        "driver": "GTiff", "height": terrain.height,
        "width": terrain.width, "count": 1, "dtype": "float32", 
        "crs": terrain.crs, "transform": transform, "nodata": -9999
    }
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(raster, 1)
    del raster

In [3]:
folder = 'wflow_model'
catchment_path = os.path.join(folder, 'inputs', 'catchment.geojson')
terrain_path = os.path.join(folder, 'inputs', 'dtm10.tif')
soil_path = os.path.join(folder, 'inputs', 'soil.geojson')
land_path = os.path.join(folder, 'inputs', 'land.geojson')
river_path = os.path.join(folder, 'inputs', 'river.geojson')
weather_path = os.path.join(folder, 'inputs', 'alesund_weather.csv')
terrain = rasterio.open(terrain_path)
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
soil = gpd.read_file(soil_path)
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)
weather = pd.read_csv(weather_path)

In [4]:
# Clip dtm to catchment
catchment_UTM = catchment.to_crs(terrain.crs)
terrain_out_path = os.path.normpath(os.path.join(folder, 'staticmaps', "dtm.tif"))
clip_catchment(catchment_UTM, terrain_path, terrain_out_path)

In [5]:
# Process river
river_UTM = river.to_crs(terrain.crs)
cols = {
    'width': (0.05, 2), 'depth': (1, 5), 'manning_n': (0.03, 0.06)
}
river_cols = river_UTM.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river_UTM[col] = pd.to_numeric(river_UTM[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river_UTM[col].isna() | (river_UTM[col] == 'None')
    river_UTM.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river_dict = {
    'river': '', 'river_width': 'width', 'river_depth': 'depth', 'river_n': 'manning_n'
}
for key, value in river_dict.items():
    river_path = os.path.normpath(os.path.join(folder, 'staticmaps', f'{key}.tif'))
    write_tif(river_path, terrain, river_UTM, value)


In [6]:
# Fix invalid soil polygon
soil_UTM = soil.to_crs(terrain.crs)
soil_cols = ['soil', 'theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
soil_UTM = fix_invalid_polygon(soil_UTM, soil_cols)
soil_UTM = soil_UTM[soil_UTM.soil != 'Water']
soil_lookup = soil_UTM.groupby("soil").agg({
    "theta_s": "first", "theta_r": "first", "k_sat_ver": "first",
    "soil_depth": "first", "conductivity_decay": "first", "brooks_corey": "first"
}).reset_index()
soil_lookup.insert(0, 'class_id', soil_lookup.index)
soil_lookup.reset_index(drop=True, inplace=True)
soil_lookup.to_csv(os.path.normpath(os.path.join(folder, 'inputs', 'soil_lookup.csv')), index=False)
# soil_out_path = os.path.normpath(os.path.join(folder, 'inputs', "soil.geojson"))
# soil_UTM.to_file(soil_out_path, driver='GeoJSON', encoding='utf-8')
soil_layers = ['theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
for value in soil_layers:
    soil_path = os.path.normpath(os.path.join(folder, 'staticmaps', f'{value}.tif'))
    write_tif(soil_path, terrain, soil_UTM, value)

In [9]:
# Fix invalid land polygon
land_UTM = land.to_crs(terrain.crs)
land_cols = ['land', 'LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
land_UTM = fix_invalid_polygon(land_UTM, land_cols)
land_lookup = land_UTM.groupby("land").agg({
    "LAI": "first", "root_depth": "first", "interception": "first",
    "manning_n": "first", "albedo": "first", "kc": "first"
}).reset_index()
for item in land_lookup['land'].values:
    df = land_UTM[land_UTM['land'] == item]
    if len(df) > 1:
        land_UTM.loc[df.index, 'id'] = str(land_lookup[land_lookup['land'] == item].index.start)
land_UTM['id'] = land_UTM['id'].astype(int)
land_path = os.path.normpath(os.path.join(folder, 'staticmaps', 'land.tif'))
write_tif(land_path, terrain, land_UTM, 'id')
land_lookup = land_lookup.drop('land', axis=1)
land_lookup.insert(0, 'class_id', land_lookup.index)
land_lookup.reset_index(drop=True, inplace=True)
land_lookup.to_csv(os.path.normpath(os.path.join(folder, 'inputs', 'land_lookup.csv')), index=False)

# land_out_path = os.path.normpath(os.path.join(folder, 'inputs', "land.geojson"))
# land_UTM.to_file(land_out_path, driver='GeoJSON', encoding='utf-8')
land_layers = ['LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
for value in land_layers:
    land_path = os.path.normpath(os.path.join(folder, 'staticmaps', f'{value}.tif'))
    write_tif(land_path, terrain, land_UTM, value)

In [ ]:
weather['datetime'] = pd.to_datetime(weather['datetime'])
weather

In [17]:
!hydromt build wflow_sbm ./wflow_model/model -i ./wflow_model/build_wflow.yml -v

2026-05-07 19:24:39,992 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-07 19:24:40,048 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-05-07 19:24:40,048 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-05-07 19:24:40,075 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-05-07 19:24:40,075 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-05-07 19:24:40,085 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-07 19:24:40,085 - hydromt.cli.main - main - ERROR - Validation of step 1 (setup_basemaps) failed because of the following error: missing a required argument: 'region'
Traceback (most recent call last):
  File "C:\Envs\hyd_ai\Lib\site-pack

Traceback (most recent call last):
  File "C:\Envs\hyd_ai\Lib\site-packages\hydromt\_utils\steps_validator.py", line 29, in _validate_steps
    _ = sig.bind(**options)
        ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vanln\AppData\Local\Programs\Python\Python311\Lib\inspect.py", line 3195, in bind
    return self._bind(args, kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vanln\AppData\Local\Programs\Python\Python311\Lib\inspect.py", line 3110, in _bind
    raise TypeError(msg) from None
TypeError: missing a required argument: 'region'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Envs\hyd_ai\Scripts\hydromt.exe\__main__.py", line 5, in <module>
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1514, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File

In [ ]:
# model.setup_basemaps(dem='dtm10')
# model.setup_rivers(river_upa=30.0, rivdph_method='powlaw')
# model.setup_soilmaps(soil_fn='soil.geojson', soil_mapping_fn='soil_lookup.csv')

In [ ]:
data_des = os.path.join(folder, 'staticmaps')
# dem = rioxr.open_rasterio(os.path.join(data_des, 'dtm.tif')).squeeze()
# thetaS = rioxr.open_rasterio(os.path.join(data_des, 'theta_s.tif')).squeeze()
# thetaR = rioxr.open_rasterio(os.path.join(data_des, 'theta_r.tif')).squeeze()
# KsatVer = rioxr.open_rasterio(os.path.join(data_des, 'k_sat_ver.tif')).squeeze()
# SoilThickness = rioxr.open_rasterio(os.path.join(data_des, 'soil_depth.tif')).squeeze()
# M = rioxr.open_rasterio(os.path.join(data_des, 'conductivity_decay.tif')).squeeze()
# c = rioxr.open_rasterio(os.path.join(data_des, 'brooks_corey.tif')).squeeze()
# landUse = rioxr.open_rasterio(os.path.join(data_des, 'land.tif')).astype('int32').squeeze()
# ds = xr.Dataset({
#     'elevation': dem, 'theta_s': thetaS, 'theta_r': thetaR, 'ksat': KsatVer,
#     'soil_thickness': SoilThickness, 'm': M, 'c': c, 'landuse': landUse
# })
# mod.set_grid(ds)

In [ ]:
# mod = WflowModel(root='model', mode='w', config_fn=os.path.join(folder, 'inputs', 'config.yaml'))
# mod.build()

In [ ]:
# print(mod.grid)